# Verify Persian G2P and Tokenizer

This notebook verifies the integration of the new `VaguyePipeline` into NeMo's G2P and Tokenizer modules.

**Prerequisites:**
Ensure you have installed the required private repositories (`pernorm`, `vaguye`, `zirneshane`, `hamnevise`) and installed NeMo in editable mode.

In [ ]:
# !pip3 install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# !pip install -e '.[all]' 

In [1]:
import logging
import os

# Configure logging
logging.basicConfig(level=logging.INFO)

from nemo.collections.tts.g2p.models.fa_ir_persian.g2p import PersianG2p
from nemo.collections.tts.g2p.models.fa_ir_persian.tokenizer import PersianPhonemesTokenizer

INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:megatron.core.msc_utils:The multistorageclient package is available.
INFO:megatron.core.distributed.fsdp.src.megatron_fsdp.megatron_fsdp:Detected Megatron Core, using Megatron-FSDP with Megatron.
INFO:megatron.core.distributed.fsdp.src.megatron_fsdp.param_and_grad_buffer:Detected Megatron Core, using Megatron-FSDP with Megatron.
INFO:nv_one_logger.exporter.export_config_manager:Final configuration contains 0 exporter(s)


## 1. Initialize G2P Module
This will load the models (Zirneshan, Hamnevise) which might take a moment.

In [2]:
try:
    # Initialize G2P
    # phoneme_dict argument is kept for compatibility but not strictly used by VaguyePipeline
    g2p = PersianG2p(phoneme_dict="dummy_path", ipa=True, stress=True)
    print("✅ PersianG2p initialized successfully!")
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    raise e

📥 Loading dictionary files from: /root/NeMo/.venv/lib/python3.12/site-packages/vaguye/persian-dict
✅ Loaded persian-primary.json
✅ Loaded persian-secondary.json
📚 Total entries loaded: 66810
⚠️ CUDA not available, using CPU
📥 Downloading default model from HuggingFace...
📂 Loading model from: /root/.cache/huggingface/hub/models--SadeghK--zirneshane/snapshots/fa0b943ba9024e24fee59b9840daf29b89960ce8/zirneshan-word-char-parsbert-embedding-classifier-v2.0.pt
✅ Model loaded successfully!
   Epoch: 10
   F1 Score: 0.7417
⚠️ CUDA not available → using CPU

🏗️  Model Architecture:
   - Shared encoder: HooshvareLab/bert-fa-base-uncased
   - Word-specific heads: 138
   - Character vocab: 67
📦 Files ready
  Model : /root/.cache/huggingface/hub/models--SadeghK--Hamnevise/snapshots/f89b2c2ac747866768c4cdd061eb6eb759e47a17/hamnevise-persian-word-disambigution-v1.0.pt
  Config: /root/.cache/huggingface/hub/models--SadeghK--Hamnevise/snapshots/f89b2c2ac747866768c4cdd061eb6eb759e47a17/tokenizer-config

## 2. Test G2P Output
Let's test with a simple Persian sentence. The output should be a list of IPA phonemes.

In [3]:
text = "سلام دنیا. من به مدرسه می‌روم."
print(f"Original Text: {text}")

phonemes = g2p(text)
print(f"Phonemes (String): {phonemes}")

Original Text: سلام دنیا. من به مدرسه می‌روم.
Phonemes (String): sæˈlɒːm donˈjɒː. mæn ˈbeː mædreˈseː ˌmiːræˈvæm.


## 3. Initialize Tokenizer
Now we pass the G2P module to the tokenizer.

In [4]:
tokenizer = PersianPhonemesTokenizer(
    g2p=g2p,
    punct=True
)
print("✅ Tokenizer initialized.")

# Check vocabulary size
print(f"Tokenizer Vocab Size: {len(tokenizer.tokens)}")

✅ Tokenizer initialized.
Tokenizer Vocab Size: 723


## 4. Test Tokenization
This step will convert text -> phonemes -> token IDs.
**Note:** If you see warnings about "unknown char/phoneme", it means the `tokenizer.py` vocabulary needs to be updated to match the new IPA symbols.

In [5]:
token_ids = tokenizer.encode(text)
print(f"Token IDs: {token_ids}")

# Decode back to check what survived
decoded = tokenizer.decode(token_ids)
print(f"Decoded (Tokens): {decoded}")

Token IDs: [32, 115, 230, 712, 108, 594, 720, 109, 32, 100, 111, 110, 712, 106, 594, 720, 46, 32, 109, 230, 110, 32, 712, 98, 101, 720, 32, 109, 230, 100, 114, 101, 712, 115, 101, 720, 32, 716, 109, 105, 720, 114, 230, 712, 118, 230, 109, 46, 32]
Decoded (Tokens):  |s|æ|ˈ|l|ɒ|ː|m| |d|o|n|ˈ|j|ɒ|ː|.| |m|æ|n| |ˈ|b|e|ː| |m|æ|d|r|e|ˈ|s|e|ː| |ˌ|m|i|ː|r|æ|ˈ|v|æ|m|.| 
